# Member 2 — Decision Tree

Owner: IT24103767

Purpose: Train and tune a Decision Tree classifier.

## Cell 1 — Load the training data

In [1]:

import pandas as pd
import numpy as np
from pathlib import Path

# Load validated data and saved split
data_path = Path("../../data/processed/hotel_bookings_validated_phase3.csv")
split_path = Path("../../results/splits/group_holdout_indices.npz")

df = pd.read_csv(data_path)
splits = np.load(split_path)

# Use the same 21 features as previous models
excluded = [
    "is_canceled", "assigned_room_type", "booking_changes",
    "deposit_type", "agent", "company", "adr",
    "required_car_parking_spaces", "total_of_special_requests",
    "reservation_status", "reservation_status_date"
]

X = df.drop(columns=excluded)
y = df["is_canceled"]

# Reuse saved training and testing indices
X_train = X.iloc[splits["train_idx"]].copy()
X_test = X.iloc[splits["test_idx"]].copy()

y_train = y.iloc[splits["train_idx"]].copy()
y_test = y.iloc[splits["test_idx"]].copy()

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("Selected features:", X_train.shape[1])
print("Training cancellation rate:", f"{y_train.mean():.2%}")


Training shape: (95334, 21)
Testing shape: (23876, 21)
Selected features: 21
Training cancellation rate: 37.01%


## Cell 2 — Decision Tree Training and Cross-Validation

In [2]:

import sys
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedGroupKFold, cross_validate
from sklearn.metrics import make_scorer, precision_score

# Import shared preprocessing
repo_root = data_path.resolve().parents[2]
sys.path.insert(0, str(repo_root))

from src.preprocessing.pipeline import build_preprocessor

# Keep identical feature combinations in the same fold
groups_train = pd.util.hash_pandas_object(X_train, index=False)

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Create Decision Tree model
tree_model = Pipeline([
    ("preprocessing", build_preprocessor(X_train)),
    ("classifier", DecisionTreeClassifier(
        max_depth=12,
        min_samples_leaf=20,
        random_state=42
    ))
])

# Evaluation metrics
scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision"
}

# Five-fold cross-validation
tree_scores = cross_validate(
    tree_model,
    X_train,
    y_train,
    groups=groups_train,
    cv=cv,
    scoring=scoring,
    n_jobs=1,
    return_train_score=True
)

print("Decision Tree - 5-Fold Cross-Validation")

for metric in scoring:
    score = tree_scores[f"test_{metric}"].mean()
    print(f"{metric}: {score:.4f}")

print("\nTraining F1-score:",
      f"{tree_scores['train_f1'].mean():.4f}")


Decision Tree - 5-Fold Cross-Validation
accuracy: 0.7875
precision: 0.7872
recall: 0.5822
f1: 0.6690
roc_auc: 0.8691
pr_auc: 0.8136

Training F1-score: 0.7172


## Cell 3 — Save Decision Tree evaluation results

In [3]:

# Store cross-validation results
metrics = list(scoring.keys())

results = pd.DataFrame({
    metric: tree_scores[f"test_{metric}"]
    for metric in metrics
})

results.index = [f"Fold {i}" for i in range(1, 6)]
results.loc["Mean"] = results.mean()

display(results.round(4))

# Save results for final model comparison
output = Path("../../results/metrics/decision_tree_cv.csv")
output.parent.mkdir(parents=True, exist_ok=True)

results.to_csv(output)

print("Results saved:", output)
print("Training F1:", round(tree_scores["train_f1"].mean(), 4))
print("Validation F1:", round(tree_scores["test_f1"].mean(), 4))


,accuracy,precision,recall,f1,roc_auc,pr_auc
Fold 1,0.8014,0.7972,0.6149,0.6943,0.8790,0.8196
Fold 2,0.7807,0.7974,0.5671,0.6628,0.8707,0.8203
Fold 3,0.7838,0.7754,0.5391,0.6360,0.8599,0.7911
Fold 4,0.7907,0.7602,0.6018,0.6718,0.8711,0.8092
Fold 5,0.7811,0.8060,0.5883,0.6801,0.8647,0.8279
Mean,0.7875,0.7872,0.5822,0.6690,0.8691,0.8136


Results saved: ..\..\results\metrics\decision_tree_cv.csv
Training F1: 0.7172
Validation F1: 0.669


## Decision Tree Evaluation Summary

The Decision Tree classifier achieved 78.75% accuracy, 78.72% precision, 58.22% recall, and an F1-score of 0.6690 using five-fold group-aware cross-validation.

The ROC-AUC of 0.8691 and PR-AUC of 0.8136 demonstrate good predictive discrimination.

Compared with Logistic Regression, Decision Tree performed better on most evaluation metrics but achieved slightly lower cancellation recall.

The training F1-score (0.7172) exceeded the validation F1-score (0.6690), indicating some overfitting. The model was therefore retained as a candidate rather than selected as the final model.

All evaluations used training data only, preserving the held-out test set for final evaluation.

## Cell 4 — Configure Hyperparameter Search

In [4]:

from sklearn.model_selection import GridSearchCV

# Hyperparameters to test
param_grid = {
    "classifier__max_depth": [8, 12, 16],
    "classifier__min_samples_leaf": [10, 30],
    "classifier__class_weight": [None, "balanced"]
}

# Use 3-fold validation for faster tuning
tuning_cv = StratifiedGroupKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

tree_search = GridSearchCV(
    estimator=tree_model,
    param_grid=param_grid,
    scoring="average_precision",
    cv=tuning_cv,
    n_jobs=1
)

print("Decision Tree hyperparameter search configured.")
print("Parameter combinations:", 12)
print("Cross-validation folds:", tuning_cv.n_splits)


Decision Tree hyperparameter search configured.
Parameter combinations: 12
Cross-validation folds: 3


## Cell 5 — Execute Decision Tree Hyperparameter Tuning

In [5]:

# Run hyperparameter tuning using training data only
tree_search.fit(
    X_train,
    y_train,
    groups=groups_train
)

print("Decision Tree tuning completed successfully.")


Decision Tree tuning completed successfully.


## Cell 6 — Display Best Hyperparameters and Tuning Score

In [6]:

# Display the best tuning results
print("Decision Tree - Hyperparameter Tuning Results\n")

print("Best hyperparameters:")
for param, value in tree_search.best_params_.items():
    print(f"{param}: {value}")

print("\nBest cross-validation PR-AUC:",
      f"{tree_search.best_score_:.4f}")


Decision Tree - Hyperparameter Tuning Results

Best hyperparameters:
classifier__class_weight: balanced
classifier__max_depth: 16
classifier__min_samples_leaf: 30

Best cross-validation PR-AUC: 0.8151


## Cell 7 — Review All Hyperparameter Combinations

In [7]:

# Review all tested configurations
tuning_results = pd.DataFrame(tree_search.cv_results_)

columns = [
    "param_classifier__max_depth",
    "param_classifier__min_samples_leaf",
    "param_classifier__class_weight",
    "mean_test_score",
    "std_test_score"
]

display(
    tuning_results[columns]
    .sort_values("mean_test_score", ascending=False)
    .round(4)
)


,param_classifier__max_depth,param_classifier__min_samples_leaf,param_classifier__class_weight,mean_test_score,std_test_score
11,16,30,balanced,0.8151,0.0051
3,12,30,None,0.8116,0.0065
9,12,30,balanced,0.8115,0.0080
8,12,10,balanced,0.8098,0.0082
5,16,30,None,0.8083,0.0069
2,12,10,None,0.8046,0.0041
10,16,10,balanced,0.7990,0.0072
1,8,30,None,0.7978,0.0104
0,8,10,None,0.7964,0.0109
4,16,10,None,0.7919,0.0076


## Cell 8 — Configure the Tuned Decision Tree

In [8]:

from sklearn.base import clone

# Create the tuned Decision Tree using the best parameters
tuned_tree_model = clone(tree_model)

tuned_tree_model.set_params(
    **tree_search.best_params_
)

print("Tuned Decision Tree configured successfully.")
print("Best parameters:", tree_search.best_params_)


Tuned Decision Tree configured successfully.
Best parameters: {'classifier__class_weight': 'balanced', 'classifier__max_depth': 16, 'classifier__min_samples_leaf': 30}


## Cell 9 — Evaluate Tuned Decision Tree

In [9]:

# Evaluate tuned Decision Tree using 5-fold cross-validation
tuned_tree_scores = cross_validate(
    tuned_tree_model,
    X_train,
    y_train,
    groups=groups_train,
    cv=cv,
    scoring=scoring,
    n_jobs=1,
    return_train_score=True
)

print("Tuned Decision Tree cross-validation completed successfully.")


Tuned Decision Tree cross-validation completed successfully.


## Cell 10 — Display Tuned Decision Tree Results

In [10]:

# Display tuned Decision Tree evaluation results
print("Tuned Decision Tree - 5-Fold Cross-Validation\n")

for metric in scoring:
    mean_score = tuned_tree_scores[f"test_{metric}"].mean()
    print(f"{metric}: {mean_score:.4f}")


Tuned Decision Tree - 5-Fold Cross-Validation

accuracy: 0.7656
precision: 0.6514
recall: 0.7907
f1: 0.7135
roc_auc: 0.8704
pr_auc: 0.8177


## Cell 11 — Compare Original and Tuned Decision Tree

In [11]:

# Compare original and tuned Decision Tree results
comparison = pd.DataFrame({
    "Original": {
        metric: tree_scores[f"test_{metric}"].mean()
        for metric in scoring
    },
    "Tuned": {
        metric: tuned_tree_scores[f"test_{metric}"].mean()
        for metric in scoring
    }
}).T

display(comparison.round(4))


,accuracy,precision,recall,f1,roc_auc,pr_auc
Original,0.7875,0.7872,0.5822,0.6690,0.8691,0.8136
Tuned,0.7656,0.6514,0.7907,0.7135,0.8704,0.8177


## Cell 12 — Check Tuned Decision Tree Overfitting

In [12]:

# Check training and validation performance
train_f1 = tuned_tree_scores["train_f1"].mean()
validation_f1 = tuned_tree_scores["test_f1"].mean()

gap = train_f1 - validation_f1

print("Tuned Decision Tree - Overfitting Check")
print(f"Training F1-score: {train_f1:.4f}")
print(f"Validation F1-score: {validation_f1:.4f}")
print(f"F1-score gap: {gap:.4f}")


Tuned Decision Tree - Overfitting Check
Training F1-score: 0.7610
Validation F1-score: 0.7135
F1-score gap: 0.0475


## Cell 13 — Save Tuned Decision Tree Evaluation Results

In [13]:

# Save tuned Decision Tree cross-validation results
tuned_results = pd.DataFrame({
    metric: tuned_tree_scores[f"test_{metric}"]
    for metric in scoring
})

tuned_results.index = [
    f"Fold {i}" for i in range(1, len(tuned_results) + 1)
]

tuned_results.loc["Mean"] = tuned_results.mean()

output = Path("../../results/metrics/decision_tree_tuned_cv.csv")
output.parent.mkdir(parents=True, exist_ok=True)

tuned_results.round(4).to_csv(output, index_label="Fold")

print("Tuned Decision Tree results saved:", output)


Tuned Decision Tree results saved: ..\..\results\metrics\decision_tree_tuned_cv.csv


## Cell 14 — Save Hyperparameter Tuning Results

In [14]:

# Save all Decision Tree hyperparameter tuning results
output = Path("../../results/metrics/decision_tree_tuning.csv")

tuning_results[[
    "param_classifier__max_depth",
    "param_classifier__min_samples_leaf",
    "param_classifier__class_weight",
    "mean_test_score",
    "std_test_score",
    "rank_test_score"
]].sort_values(
    "rank_test_score"
).to_csv(output, index=False)

print("Decision Tree tuning results saved:", output)
print("Best parameters:", tree_search.best_params_)


Decision Tree tuning results saved: ..\..\results\metrics\decision_tree_tuning.csv
Best parameters: {'classifier__class_weight': 'balanced', 'classifier__max_depth': 16, 'classifier__min_samples_leaf': 30}


## Cell 15 — Final Decision Tree Conclusion


## Hyperparameter Tuning Summary

Hyperparameter tuning was performed using GridSearchCV with three-fold StratifiedGroupKFold cross-validation, optimizing PR-AUC on the training data.

The selected Decision Tree configuration was:
- `max_depth = 16`
- `min_samples_leaf = 30`
- `class_weight = "balanced"`

In five-fold cross-validation, tuning improved recall from 0.5822 to 0.7907, F1-score from 0.6690 to 0.7135, and PR-AUC from 0.8136 to 0.8177.

Precision decreased from 0.7872 to 0.6514, indicating a trade-off between identifying more cancellations and generating additional false cancellation alerts.

The training–validation F1 gap remained similar (0.0482 before tuning and 0.0475 after tuning).

**Decision:** Retain the tuned Decision Tree as the preferred Decision Tree candidate for final model comparison. Final selection will depend on comparison with the other models and evaluation on the untouched test set.
